In [1]:
import scipp as sc
import scippnexus as snx
import plopp as pp
import numpy as np

from easydynamics.Job import Job
from easydynamics.Experiment import Experiment
from easydynamics.Experiment import Data

from easydynamics.sample import BrownianTranslationalDiffusion
from easydynamics.sample import JumpDiffusion

from easydynamics.sample import SampleModel
from easydynamics.sample import Lorentzian
from easydynamics.sample import DeltaFunction
from easydynamics.sample import Polynomial
# from easydynamics.sample import DampedHarmonicOscillator


from easydynamics.sample import Gaussian



%matplotlib widget
data_path = r"C:\Users\henrikjacobsen3\Dropbox\DMSC\Halric 2025\new_IRIS_data\iris_Cells_Plus_TCZ_108330_to108389_Plus_Empty_Data_SQW"



In [2]:
filenumber = 108556 # Empty container long

filename = data_path + fr"/iris{filenumber}_graphite002_sqw.nxs"
all_data = snx.load(filename)

c:\Users\henrikjacobsen3\AppData\Local\miniforge3\envs\newdynamics\Lib\site-packages\scippnexus\base.py:387: UserWarning: Failed to load /mantid_workspace_1/instrument/detector as NXdetector: Could not determine signal field or dimensions. Falling back to loading HDF5 group children as scipp.DataGroup.
  self._warn_fallback(e)
c:\Users\henrikjacobsen3\AppData\Local\miniforge3\envs\newdynamics\Lib\site-packages\scippnexus\base.py:387: UserWarning: Failed to load /mantid_workspace_1/instrument/physical_detectors as NXdetector: Could not determine signal field or dimensions. Falling back to loading HDF5 group children as scipp.DataGroup.
  self._warn_fallback(e)
c:\Users\henrikjacobsen3\AppData\Local\miniforge3\envs\newdynamics\Lib\site-packages\scippnexus\base.py:387: UserWarning: Failed to load /mantid_workspace_1/instrument/physical_monitors as NXmonitor: Could not determine signal field or dimensions. Falling back to loading HDF5 group children as scipp.DataGroup.
  self._warn_fallbac

In [3]:
all_data

DataGroup(sizes={'dim_0': None, 'dim_1': None, 'detector_number': 110, 'time': None, 'axis2': 27, 'axis1': 2000}, keys=[
    mantid_workspace_1: DataGroup(9, {'dim_0': None, 'dim_1': None, 'detector_number': 110, 'time': None, 'axis2': 27, 'axis1': 2000}),
])

In [4]:
# Load data function
number_of_Q_bins = 10 # we may change this later, but this seems like a reasonable start
number_of_energy_bins = 501

def load_data(filenumber: int, number_of_Q_bins, number_of_energy_bins):
    filename = data_path + fr"/iris{filenumber}_graphite002_sqw.nxs"
    all_data = snx.load(filename)

    temperature = all_data['mantid_workspace_1']['logs']['Sample']
    temperature.coords['time'].unit = 's'
    temperature.unit = 'K'

    data=all_data['mantid_workspace_1']['workspace'].rename({'axis1': 'energy', 'axis2': 'Q'})

    data.coords['energy'].unit = 'meV'
    data.coords['Q'].unit = '1/Angstrom'

    del data.coords['frac_area']

    # Rebin data and use midpoints instead of edges
    rebinned_data=data.rebin(Q=number_of_Q_bins, energy=number_of_energy_bins)

    rebinned_data.coords['Q'] = sc.midpoints(rebinned_data.coords['Q'])
    rebinned_data.coords['energy'] = sc.midpoints(rebinned_data.coords['energy'])


    # Remove 0 variances
    v = rebinned_data.variances
    v[v <= 0] = 1.0


    return rebinned_data, temperature




In [5]:
# filenumbers = range(108330,108390)
# # for filenumber in filenumbers:
# #     data, temperature = load_data(filenumber)
# #     average_tempreature = sc.mean(temperature)


# datas = []
# temps = []

# for filenumber in filenumbers:
#     data, temperature = load_data(filenumber,number_of_Q_bins,number_of_energy_bins)

#     # average temperature for this file
#     avg_T = sc.mean(temperature)
#     print(filenumber, avg_T)
#     datas.append(data)
#     temps.append(avg_T)
    
# temperature = sc.array(dims=['temperature'], values = np.asarray([t.value for t in temps]), unit = 'K')
# # Combine all data arrays into a single DataArray along a new dimension "temperature"
# all_data = sc.concat(datas, dim='temperature')
# all_data.coords['temperature'] = temperature


In [6]:

# all_data = sc.sort(all_data,'temperature')

In [7]:
# pp.slicer(all_data,keep=['energy'])

In [8]:
# all_data

In [9]:
# Now let's use the low temperature data to estimate the resolution.
resolution_job= Job(name='resolution')


filenumber = 108331 # base T long
lowt_data,lowt_temperature = load_data(filenumber,number_of_Q_bins,number_of_energy_bins)

exp=Experiment()
res_data=Data()
res_data.append(lowt_data)

exp.set_data(res_data)

resolution_job.set_experiment(exp)


bg=SampleModel('Background')
bg.add_component(Polynomial(coefficients=[1e-3]))
resolution_job.set_background_model(bg)


resolution_model=SampleModel(name="ResolutionModel")
resolution_model.add_component(Gaussian(name="Res1", area=1.5,width=0.01))
resolution_model.add_component(Gaussian(name="Res2", area=1.0,width=0.015,center=-0.01))
resolution_model.add_component(Lorentzian(name="Res3", area=0.3,width=0.015,center=-0.025))

resolution_job.set_theory(resolution_model)
resolution_job.generate_analysis_for_cuts()
for i in range(len(resolution_job.analysis)):
    resolution_job.analysis[i].get_fit_parameters()[8].min=0 #polynomial
    resolution_job.analysis[i].get_fit_parameters()[0].min=0 #Gauss 1 area
    resolution_job.analysis[i].get_fit_parameters()[2].min=0 #Gauss 2 area

resolution_job.fit(sequential = "Q")


c:\Users\henrikjacobsen3\AppData\Local\miniforge3\envs\newdynamics\Lib\site-packages\scippnexus\base.py:387: UserWarning: Failed to load /mantid_workspace_1/instrument/detector as NXdetector: Could not determine signal field or dimensions. Falling back to loading HDF5 group children as scipp.DataGroup.
  self._warn_fallback(e)
c:\Users\henrikjacobsen3\AppData\Local\miniforge3\envs\newdynamics\Lib\site-packages\scippnexus\base.py:387: UserWarning: Failed to load /mantid_workspace_1/instrument/physical_detectors as NXdetector: Could not determine signal field or dimensions. Falling back to loading HDF5 group children as scipp.DataGroup.
  self._warn_fallback(e)
c:\Users\henrikjacobsen3\AppData\Local\miniforge3\envs\newdynamics\Lib\site-packages\scippnexus\base.py:387: UserWarning: Failed to load /mantid_workspace_1/instrument/physical_monitors as NXmonitor: Could not determine signal field or dimensions. Falling back to loading HDF5 group children as scipp.DataGroup.
  self._warn_fallbac

In [10]:

resolution_job.plot_data_and_model(intensity_min=0.0, intensity_max=120.0,
                            energy_min=-0.2, energy_max=0.2)

InteractiveFigure(children=(HBar(), HBar(children=(VBar(children=(Toolbar(children=(ButtonTool(icon='home', la…

In [11]:
resolution_job._analysis[1].get_fit_parameters()

a=resolution_job.return_data_and_model()
a

DataGroup(sizes={'Q': 10, 'energy': 501}, keys=[
    Data: DataArray({'Q': 10, 'energy': 501}),
    Model: DataArray({'Q': 10, 'energy': 501}),
    Res1: DataArray({'Q': 10, 'energy': 501}),
    Res2: DataArray({'Q': 10, 'energy': 501}),
    Res3: DataArray({'Q': 10, 'energy': 501}),
    Polynomial: DataArray({'Q': 10, 'energy': 501}),
])

In [12]:

def fit_simultaneous(filenumber,number_of_Q_bins,number_of_energy_bins):
    data, temperature = load_data(filenumber,number_of_Q_bins,number_of_energy_bins)
    Cells_Plus_TCZ_simultaneous= Job(name='JumpDiffusion')


    exp=Experiment()
    highT_data_simul=Data()
    highT_data_simul.append(data)

    exp.set_data(highT_data_simul)

    Cells_Plus_TCZ_simultaneous.set_experiment(exp)
    Cells_Plus_TCZ_simultaneous.generate_empty_analysis_array()


    bg=SampleModel('Background')
    bg.add_component(Polynomial(coefficients=[1e-3]))
    Cells_Plus_TCZ_simultaneous.set_background_model(bg)
    Cells_Plus_TCZ_simultaneous.set_background_model_for_all_analyses()
    T_motion=249.1
    avg_T = sc.mean(temperature)
    if avg_T.value>T_motion:
        diffusion_model=JumpDiffusion(name="JumpDiffusion", diffusion_coefficient=1.8, tau = 5.0,scale=0.2)
        Cells_Plus_TCZ_simultaneous.set_diffusion_model(diffusion_model)
    # delta_model=DeltaFunction(name="Delta",area=0.05)
    # Cells_Plus_TCZ_simultaneous.set_theory_for_all_analyses(delta_model) # Something wrong here, it uses the same parameter
    # Cells_Plus_TCZ.use_fit_as_resolution(resolution_job) # doesn't seem to work for some reason
    for i in range(len(Cells_Plus_TCZ_simultaneous._analysis)):
        this_resolution_model=resolution_job._analysis[i]._theory
        Cells_Plus_TCZ_simultaneous._analysis[i].set_resolution_model(this_resolution_model)
        Cells_Plus_TCZ_simultaneous._analysis[i].fix_resolution_parameters()
        if avg_T.value>T_motion:
            Cells_Plus_TCZ_simultaneous._analysis[i]._theory.add_component(DeltaFunction(name="Delta",area=0.05))
            Cells_Plus_TCZ_simultaneous.analysis[i].get_fit_parameters()[5].min=1e-8 # BG
            Cells_Plus_TCZ_simultaneous.analysis[i].get_fit_parameters()[2].min=0.001   #D     
            Cells_Plus_TCZ_simultaneous.analysis[i].get_fit_parameters()[0].min=0.0  #scale      
            Cells_Plus_TCZ_simultaneous.analysis[i].get_fit_parameters()[3].min=0.001  #tau 
        else:
            theory=SampleModel()
            theory.add_component(DeltaFunction(name="Delta",area=1.85))
            Cells_Plus_TCZ_simultaneous._analysis[i].set_theory(theory)
            Cells_Plus_TCZ_simultaneous.analysis[i].get_fit_parameters()[1].min=1e-8 # BG

    result=Cells_Plus_TCZ_simultaneous.fit_simultaneous()

    return Cells_Plus_TCZ_simultaneous,result,temperature


def fit_sequential(filenumber,number_of_Q_bins,number_of_energy_bins):
    data, temperature = load_data(filenumber,number_of_Q_bins,number_of_energy_bins)
    Cells_Plus_TCZ_simultaneous= Job(name='JumpDiffusion')


    exp=Experiment()
    highT_data_simul=Data()
    highT_data_simul.append(data)

    exp.set_data(highT_data_simul)

    Cells_Plus_TCZ_simultaneous.set_experiment(exp)
    Cells_Plus_TCZ_simultaneous.generate_empty_analysis_array()


    bg=SampleModel('Background')
    bg.add_component(Polynomial(coefficients=[1e-3]))
    Cells_Plus_TCZ_simultaneous.set_background_model(bg)
    Cells_Plus_TCZ_simultaneous.set_background_model_for_all_analyses()
    T_motion=249.1
    avg_T = sc.mean(temperature)
    # if avg_T.value>T_motion:
        # diffusion_model=JumpDiffusion(name="JumpDiffusion", diffusion_coefficient=1.8, tau = 5.0,scale=0.2)
        # Cells_Plus_TCZ_simultaneous.set_diffusion_model(diffusion_model)
    # delta_model=DeltaFunction(name="Delta",area=0.05)
    # Cells_Plus_TCZ_simultaneous.set_theory_for_all_analyses(delta_model) # Something wrong here, it uses the same parameter
    # Cells_Plus_TCZ.use_fit_as_resolution(resolution_job) # doesn't seem to work for some reason
    for i in range(len(Cells_Plus_TCZ_simultaneous._analysis)):
        this_resolution_model=resolution_job._analysis[i]._theory
        Cells_Plus_TCZ_simultaneous._analysis[i].set_resolution_model(this_resolution_model)
        Cells_Plus_TCZ_simultaneous._analysis[i].fix_resolution_parameters()
        if avg_T.value>T_motion:
            theory=SampleModel()
            theory.add_component(DeltaFunction(name="Delta",area=0.05))
            theory.add_component(Lorentzian(name="Lorentzian",area=0.3, width=0.1))
            Cells_Plus_TCZ_simultaneous._analysis[i].set_theory(theory)
            # Cells_Plus_TCZ_simultaneous._analysis[i]._theory.add_component(DeltaFunction(name="Delta",area=0.05))
            # Cells_Plus_TCZ_simultaneous._analysis[i]._theory.add_component(Lorentzian(name="Lorentzian",area=0.3, width=0.1))
            Cells_Plus_TCZ_simultaneous.analysis[i].get_fit_parameters()[3].min=1e-10 # BG
            # Cells_Plus_TCZ_simultaneous.analysis[i].get_fit_parameters()[2].min=0.001   #D     
            # Cells_Plus_TCZ_simultaneous.analysis[i].get_fit_parameters()[0].min=0.0  #scale      
            # Cells_Plus_TCZ_simultaneous.analysis[i].get_fit_parameters()[3].min=0.001  #tau 
        else:
            theory=SampleModel()
            theory.add_component(DeltaFunction(name="Delta",area=1.85))
            theory.add_component(Lorentzian(name="Lorentzian",area=0.0, width=1.0))
            Cells_Plus_TCZ_simultaneous._analysis[i].set_theory(theory)
            # Cells_Plus_TCZ_simultaneous.analysis[i].get_fit_parameters()[1].min=1e-10 # BG
            Cells_Plus_TCZ_simultaneous.analysis[i].get_fit_parameters()[3].min=1e-10 # BG
            Cells_Plus_TCZ_simultaneous.analysis[i].get_fit_parameters()[1].fixed=True # don't fit the Lorentzian at low temperature, but keep it for plotting purposes
            Cells_Plus_TCZ_simultaneous.analysis[i].get_fit_parameters()[2].fixed=True # don't fit the Lorentzian at low temperature, but keep it for plotting purposes

    result=Cells_Plus_TCZ_simultaneous.fit()

    return Cells_Plus_TCZ_simultaneous,result,temperature


In [13]:
this_job,this_result,this_temperature = fit_sequential(filenumber,number_of_Q_bins,number_of_energy_bins)


c:\Users\henrikjacobsen3\AppData\Local\miniforge3\envs\newdynamics\Lib\site-packages\scippnexus\base.py:387: UserWarning: Failed to load /mantid_workspace_1/instrument/detector as NXdetector: Could not determine signal field or dimensions. Falling back to loading HDF5 group children as scipp.DataGroup.
  self._warn_fallback(e)
c:\Users\henrikjacobsen3\AppData\Local\miniforge3\envs\newdynamics\Lib\site-packages\scippnexus\base.py:387: UserWarning: Failed to load /mantid_workspace_1/instrument/physical_detectors as NXdetector: Could not determine signal field or dimensions. Falling back to loading HDF5 group children as scipp.DataGroup.
  self._warn_fallback(e)
c:\Users\henrikjacobsen3\AppData\Local\miniforge3\envs\newdynamics\Lib\site-packages\scippnexus\base.py:387: UserWarning: Failed to load /mantid_workspace_1/instrument/physical_monitors as NXmonitor: Could not determine signal field or dimensions. Falling back to loading HDF5 group children as scipp.DataGroup.
  self._warn_fallbac

In [14]:
this_job._analysis[0].get_fit_parameters()

[<Parameter 'Delta area': 2.8541 meV, bounds=[0.0:inf]>,
 <Parameter 'Lorentzian width': 1.0000 meV, bounds=[0.0:inf]>,
 <Parameter 'offset': -0.0026 meV, bounds=[-inf:inf]>]

In [15]:
filenumbers = range(108330,108390)
job_list=[]
temperature_list=[]
number_of_Q_bins = 10 # we may change this later, but this seems like a reasonable start
number_of_energy_bins = 501
for filenumber in filenumbers:
    this_job,this_result,this_temperature = fit_sequential(filenumber,number_of_Q_bins,number_of_energy_bins)
    job_list.append(this_job)
    temperature_list.append(this_temperature)
    print(this_temperature)
    print(this_job.analysis[5].get_fit_parameters())


this_job.plot_data_and_model_residual(intensity_min=-0.5, intensity_max=5,
                            energy_min=-0.3, energy_max=0.5)

c:\Users\henrikjacobsen3\AppData\Local\miniforge3\envs\newdynamics\Lib\site-packages\scippnexus\base.py:387: UserWarning: Failed to load /mantid_workspace_1/instrument/detector as NXdetector: Could not determine signal field or dimensions. Falling back to loading HDF5 group children as scipp.DataGroup.
  self._warn_fallback(e)
c:\Users\henrikjacobsen3\AppData\Local\miniforge3\envs\newdynamics\Lib\site-packages\scippnexus\base.py:387: UserWarning: Failed to load /mantid_workspace_1/instrument/physical_detectors as NXdetector: Could not determine signal field or dimensions. Falling back to loading HDF5 group children as scipp.DataGroup.
  self._warn_fallback(e)
c:\Users\henrikjacobsen3\AppData\Local\miniforge3\envs\newdynamics\Lib\site-packages\scippnexus\base.py:387: UserWarning: Failed to load /mantid_workspace_1/instrument/physical_monitors as NXmonitor: Could not determine signal field or dimensions. Falling back to loading HDF5 group children as scipp.DataGroup.
  self._warn_fallbac

<scipp.DataArray>
Dimensions: Sizes[time:121, ]
Coordinates:
* time                      float64              [s]  (time)  [0, 9, ..., 7284, 7450]
Data:
                            float64              [K]  (time)  [15.0555, 14.9733, ..., 10.093, 10.093]


[<Parameter 'Delta area': 2.8778 meV, bounds=[0.0:inf]>, <Parameter 'Lorentzian width': 1.0000 meV, bounds=[0.0:inf]>, <Parameter 'offset': -0.0020 meV, bounds=[-inf:inf]>]


c:\Users\henrikjacobsen3\AppData\Local\miniforge3\envs\newdynamics\Lib\site-packages\scippnexus\base.py:387: UserWarning: Failed to load /mantid_workspace_1/instrument/detector as NXdetector: Could not determine signal field or dimensions. Falling back to loading HDF5 group children as scipp.DataGroup.
  self._warn_fallback(e)
c:\Users\henrikjacobsen3\AppData\Local\miniforge3\envs\newdynamics\Lib\site-packages\scippnexus\base.py:387: UserWarning: Failed to load /mantid_workspace_1/instrument/physical_detectors as NXdetector: Could not determine signal field or dimensions. Falling back to loading HDF5 group children as scipp.DataGroup.
  self._warn_fallback(e)
c:\Users\henrikjacobsen3\AppData\Local\miniforge3\envs\newdynamics\Lib\site-packages\scippnexus\base.py:387: UserWarning: Failed to load /mantid_workspace_1/instrument/physical_monitors as NXmonitor: Could not determine signal field or dimensions. Falling back to loading HDF5 group children as scipp.DataGroup.
  self._warn_fallbac

<scipp.DataArray>
Dimensions: Sizes[time:91, ]
Coordinates:
* time                      float64              [s]  (time)  [0, 8, ..., 33308, 33757]
Data:
                            float64              [K]  (time)  [10.093, 10.093, ..., 10.0471, 10.0471]


[<Parameter 'Delta area': 2.8749 meV, bounds=[0.0:inf]>, <Parameter 'Lorentzian width': 1.0000 meV, bounds=[0.0:inf]>, <Parameter 'offset': -0.0019 meV, bounds=[-inf:inf]>]


c:\Users\henrikjacobsen3\AppData\Local\miniforge3\envs\newdynamics\Lib\site-packages\scippnexus\base.py:387: UserWarning: Failed to load /mantid_workspace_1/instrument/detector as NXdetector: Could not determine signal field or dimensions. Falling back to loading HDF5 group children as scipp.DataGroup.
  self._warn_fallback(e)
c:\Users\henrikjacobsen3\AppData\Local\miniforge3\envs\newdynamics\Lib\site-packages\scippnexus\base.py:387: UserWarning: Failed to load /mantid_workspace_1/instrument/physical_detectors as NXdetector: Could not determine signal field or dimensions. Falling back to loading HDF5 group children as scipp.DataGroup.
  self._warn_fallback(e)
c:\Users\henrikjacobsen3\AppData\Local\miniforge3\envs\newdynamics\Lib\site-packages\scippnexus\base.py:387: UserWarning: Failed to load /mantid_workspace_1/instrument/physical_monitors as NXmonitor: Could not determine signal field or dimensions. Falling back to loading HDF5 group children as scipp.DataGroup.
  self._warn_fallbac

...

c:\Users\henrikjacobsen3\AppData\Local\miniforge3\envs\newdynamics\Lib\site-packages\scippnexus\base.py:387: UserWarning: Failed to load /mantid_workspace_1/instrument/detector as NXdetector: Could not determine signal field or dimensions. Falling back to loading HDF5 group children as scipp.DataGroup.
  self._warn_fallback(e)
c:\Users\henrikjacobsen3\AppData\Local\miniforge3\envs\newdynamics\Lib\site-packages\scippnexus\base.py:387: UserWarning: Failed to load /mantid_workspace_1/instrument/physical_detectors as NXdetector: Could not determine signal field or dimensions. Falling back to loading HDF5 group children as scipp.DataGroup.
  self._warn_fallback(e)
c:\Users\henrikjacobsen3\AppData\Local\miniforge3\envs\newdynamics\Lib\site-packages\scippnexus\base.py:387: UserWarning: Failed to load /mantid_workspace_1/instrument/physical_monitors as NXmonitor: Could not determine signal field or dimensions. Falling back to loading HDF5 group children as scipp.DataGroup.
  self._warn_fallbac

<scipp.DataArray>
Dimensions: Sizes[time:57, ]
Coordinates:
* time                      float64              [s]  (time)  [0, 28, ..., 6969, 7615]
Data:
                            float64              [K]  (time)  [285.059, 288.293, ..., 289.959, 289.959]


[<Parameter 'Delta area': 0.0425 ± 0.0016 meV, bounds=[0.0:inf]>, <Parameter 'Lorentzian area': 0.4618 ± 0.0042 meV, bounds=[-inf:inf]>, <Parameter 'Lorentzian width': 0.1198 ± 0.0019 meV, bounds=[0.0:inf]>, <Parameter 'Polynomial_c0': 1.352e-10 ± 3.792e-04, bounds=[1e-10:inf]>, <Parameter 'offset': -0.0033 ± 0.0004 meV, bounds=[-inf:inf]>]


c:\Users\henrikjacobsen3\AppData\Local\miniforge3\envs\newdynamics\Lib\site-packages\scippnexus\base.py:387: UserWarning: Failed to load /mantid_workspace_1/instrument/detector as NXdetector: Could not determine signal field or dimensions. Falling back to loading HDF5 group children as scipp.DataGroup.
  self._warn_fallback(e)
c:\Users\henrikjacobsen3\AppData\Local\miniforge3\envs\newdynamics\Lib\site-packages\scippnexus\base.py:387: UserWarning: Failed to load /mantid_workspace_1/instrument/physical_detectors as NXdetector: Could not determine signal field or dimensions. Falling back to loading HDF5 group children as scipp.DataGroup.
  self._warn_fallback(e)
c:\Users\henrikjacobsen3\AppData\Local\miniforge3\envs\newdynamics\Lib\site-packages\scippnexus\base.py:387: UserWarning: Failed to load /mantid_workspace_1/instrument/physical_monitors as NXmonitor: Could not determine signal field or dimensions. Falling back to loading HDF5 group children as scipp.DataGroup.
  self._warn_fallbac

<scipp.DataArray>
Dimensions: Sizes[time:3, ]
Coordinates:
* time                      float64              [s]  (time)  [0, 69, 1106]
Data:
                            float64              [K]  (time)  [289.959, 289.959, 289.959]


[<Parameter 'Delta area': 0.0406 meV, bounds=[0.0:inf]>, <Parameter 'Lorentzian area': 0.1166 meV, bounds=[-inf:inf]>, <Parameter 'Lorentzian width': 0.0573 meV, bounds=[0.0:inf]>, <Parameter 'Polynomial_c0': 1.015e-10, bounds=[1e-10:inf]>, <Parameter 'offset': -0.0017 meV, bounds=[-inf:inf]>]


InteractiveFigure(children=(HBar(), HBar(children=(VBar(children=(Toolbar(children=(ButtonTool(icon='home', la…

In [16]:
job_list[54].plot_data_and_model_residual(intensity_min=-0.5, intensity_max=5,
                            energy_min=-0.3, energy_max=0.5)

InteractiveFigure(children=(HBar(), HBar(children=(VBar(children=(Toolbar(children=(ButtonTool(icon='home', la…

In [17]:
job_list[49]._analysis[5].get_fit_parameters()
# job_list[49].get_parameters_as_data_group()

[<Parameter 'Delta area': 1.7780 ± 0.0107 meV, bounds=[0.0:inf]>,
 <Parameter 'Lorentzian area': 0.2312 ± 0.0091 meV, bounds=[-inf:inf]>,
 <Parameter 'Lorentzian width': 0.0266 ± 0.0017 meV, bounds=[0.0:inf]>,
 <Parameter 'Polynomial_c0': 0.0121 ± 0.0006, bounds=[1e-10:inf]>,
 <Parameter 'offset': -0.0016 ± 0.0000 meV, bounds=[-inf:inf]>]

In [18]:
    
# temperature = sc.array(dims=['temperature'], values = np.asarray([t.value for t in temps]), unit = 'K')
# # Combine all data arrays into a single DataArray along a new dimension "temperature"
# all_data = sc.concat(datas, dim='temperature')
# all_data.coords['temperature'] = temperature


In [19]:
# P1=job_list[49].get_parameters_as_data_group()
# P2=job_list[50].get_parameters_as_data_group()
# P=sc.concat([P1,P2],dim='temperature')
groups = [job.get_parameters_as_data_group() for job in job_list]
fit_parameters = sc.concat(groups, dim='temperature')
avg_temperature=[sc.mean(t) for t in temperature_list]
temperature = sc.array(dims=['temperature'], values = np.asarray([t.value for t in avg_temperature]), unit = 'K')

# fit_parameters.coords['temperature']=temperature

In [20]:
a=fit_parameters['Delta area']['value']
a
a.coords['temperature']=temperature
pp.slicer(a,vmin=0,vmax=3)

c:\Users\henrikjacobsen3\AppData\Local\miniforge3\envs\newdynamics\Lib\site-packages\plopp\plotting\common.py:302: RuntimeWarning: The input contains a coordinate with unsorted values (temperature). The results may be unpredictable. Coordinates can be sorted using `scipp.sort(data, dim="to_be_sorted", order="ascending")`.
  _check_coord_sanity(out)


InteractiveFigure(children=(HBar(), HBar(children=(VBar(children=(Toolbar(children=(ButtonTool(icon='home', la…

In [21]:
width=fit_parameters['Lorentzian width']['value']
width.coords['temperature']=temperature

area=fit_parameters['Lorentzian area']['value']
area.coords['temperature']=temperature


In [22]:
pp.slicer(width,vmin=0,vmax=0.2)

c:\Users\henrikjacobsen3\AppData\Local\miniforge3\envs\newdynamics\Lib\site-packages\plopp\plotting\common.py:302: RuntimeWarning: The input contains a coordinate with unsorted values (temperature). The results may be unpredictable. Coordinates can be sorted using `scipp.sort(data, dim="to_be_sorted", order="ascending")`.
  _check_coord_sanity(out)


InteractiveFigure(children=(HBar(), HBar(children=(VBar(children=(Toolbar(children=(ButtonTool(icon='home', la…

In [23]:
pp.slicer(area)

c:\Users\henrikjacobsen3\AppData\Local\miniforge3\envs\newdynamics\Lib\site-packages\plopp\plotting\common.py:302: RuntimeWarning: The input contains a coordinate with unsorted values (temperature). The results may be unpredictable. Coordinates can be sorted using `scipp.sort(data, dim="to_be_sorted", order="ascending")`.
  _check_coord_sanity(out)


InteractiveFigure(children=(HBar(), HBar(children=(VBar(children=(Toolbar(children=(ButtonTool(icon='home', la…

In [24]:
job_list[50].plot_data_and_model(intensity_min=-0.5, intensity_max=80,
                            energy_min=-0.3, energy_max=0.5)

InteractiveFigure(children=(HBar(), HBar(children=(VBar(children=(Toolbar(children=(ButtonTool(icon='home', la…

In [25]:
# resolution_job.return_data_and_model()

groups = [job.return_data_and_model() for job in job_list]
fit_parameters = sc.concat(groups, dim='temperature')
# avg_temperature=[sc.mean(t) for t in temperature_list]
# temperature = sc.array(dims=['temperature'], values = np.asarray([t.value for t in avg_temperature]), unit = 'K')

linestyle = {"Data": "none", "Model": "-", "Lorentzian": "--", "Polynomial":"--", "Delta":"--"}
marker = {"Data": "o", "Model": "none", "Lorentzian": "none", "Polynomial":"none", "Delta":"none"}
markerfacecolor = {"Data": "none", "Model": "none", "Lorentzian": "none", "Polynomial":"none", "Delta":"none"}
color = {"Data": "black", "Model": "red", "Lorentzian": "green", "Polynomial":"purple", "Delta": "blue"}


pp.slicer(fit_parameters,keep='energy',vmin=-0.5,vmax=20,linestyle=linestyle, marker=marker,markerfacecolor=markerfacecolor, color=color)
# fit_parameters

InteractiveFigure(children=(HBar(), HBar(children=(VBar(children=(Toolbar(children=(ButtonTool(icon='home', la…